### Import des librairies utiles pour ce projet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### Chargement du dataset selon les csv remis lors de la mission

In [ ]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

In [ ]:
print('Les données minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les données maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

In [ ]:
# Conversion de la colonne date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [ ]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [ ]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [ ]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [ ]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

In [ ]:
# Grahique concernant la saisonnaluté du CA. Pas de réelle saisonalité
# Interprétation limitée et tendance à une certaine stabilité - saisonnalité marquée mais pas de pics

figure = px.line(
    df_ca, 
    x='date', 
    y=['price', 'Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=px.colors.qualitative.Pastel,
    markers=False,
    title='Évolution du chiffre d\'affaires',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

#### Chiffre d’affaires par catégorie + évolution dans le temps

In [ ]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [ ]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

In [ ]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

In [ ]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ', 
    markers=True,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

#### Nombre de clients par mois + évolution dans le temps

In [ ]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

In [ ]:
#Affichage du nombre de clients uniques par mois
df_clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
df_clients_par_mois.head()

In [ ]:
figure3=px.line(
    df_clients_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [ ]:
#Affichage du nombre de clients par mois
df_clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
df_clients_total_par_mois.head()

In [ ]:
figure4=px.line(
    df_clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

#### Nombre de transactions + évolution dans le temps

In [ ]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

In [ ]:
#Affichage du nombre de transactions par semaine
df_transactions_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['session_id'].count().reset_index(name='Nombre de transactions par semaine'))
df_transactions_semaine.head()

In [ ]:
#Affichage du nombre de transactions par mois
df_transactions_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['session_id'].count().reset_index(name='Nombre de transactions par mois'))
df_transactions_mois.head()

In [ ]:
#Affichage du nombre de transactions par trimestre
df_transactions_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['session_id'].count().reset_index(name='Nombre de transactions par trimestre'))
df_transactions_trimestre.head()

### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

In [ ]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_transactions= df_transactions_semaine.merge(df_transactions_mois, on = 'date', how='outer').sort_values('date')
df_transactions= df_transactions.merge(df_transactions_trimestre, on = 'date', how='outer').sort_values('date')
df_transactions['Nombre de transactions par mois'] = (df_transactions['Nombre de transactions par mois'].interpolate(method='linear'))
df_transactions['Nombre de transactions par trimestre'] = (df_transactions['Nombre de transactions par trimestre'].interpolate(method='linear'))
df_transactions.head()

In [ ]:
figure5=px.line(
    df_transactions,
    x='date',
    y=[ 'Nombre de transactions par semaine','Nombre de transactions par mois','Nombre de transactions par trimestre'],
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure5.show()

#### Nombre de produits vendus + évolution dans le temps

In [ ]:
# tous les produits sur toute la période, puis par mois, puis par semaine
#Affichage du nombre de produits vendus par semaine
df_lapage_prod_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par semaine'))
df_lapage_prod_semaine.head()

In [ ]:
#Affichage du nombre de produits vendus par mois
df_lapage_prod_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par mois'))

In [ ]:
#Affichage du nombre de produits vendus par trimestre
df_lapage_prod_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par trimestre'))

In [ ]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_lapage_prod= df_lapage_prod_semaine.merge(df_lapage_prod_mois, on = 'date', how='outer').sort_values('date')
df_lapage_prod= df_lapage_prod.merge(df_lapage_prod_trimestre, on = 'date', how='outer').sort_values('date')
df_lapage_prod['Nombre de produits vendus par mois'] = (df_lapage_prod['Nombre de produits vendus par mois'].interpolate(method='linear'))
df_lapage_prod['Nombre de produits vendus par trimestre'] = (df_lapage_prod['Nombre de produits vendus par trimestre'].interpolate(method='linear'))
df_lapage_prod.head()

In [ ]:
figure6=px.line(
    df_lapage_prod,
    x='date',
    y=[ 'Nombre de produits vendus par semaine','Nombre de produits vendus par mois','Nombre de produits vendus par trimestre'],
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

# Top, Flop, la répartition par catégorie

- les tops,
- les flops,
- la répartition par catégorie

In [ ]:
# Création d'une colonne dans le df principal, indiquant la quantité de produits vendus, par article
df_lapage['quantity'] = df_lapage.groupby('id_prod')['id_prod'].transform('count')
df_lapage.head()

In [ ]:
# Création d'un dataframe restreint contenant la colonne quantité
df_quantity = df_lapage[['id_prod', 'price', 'categ', 'quantity']]
df_quantity.head()

In [ ]:
# Eviter le warning concernant les bugs de pandas sur les copies de dataframes
df_quantity = df_quantity.copy()

df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']

In [ ]:
# Les tops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=False).head(10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])


In [ ]:
# Graphique du top 10 

In [ ]:
# Les flops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=True).head(10)
    print(f'Le top 10 des produits les moins vendus de la catégorie {categ} est le suivant :')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])


In [ ]:
#### Utiliser le subplot de plotly

#### Profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.


# IDENTIFICATION CLIENTS B TO B ET B TO C

In [ ]:
# Il existe plusieurs moyens pour identifier les B to B : 

In [ ]:
# # CA très élevé et dans ce cas nous identifions 4 clients ayant généré plusieurs centaines de milliers d'euros de CA
ca_par_client = df_lapage.groupby(['client_id'])['price'].sum().reset_index().sort_values('price', ascending=False)
ca_par_client.head(10)
# clients = ['c_1609','c_4958','c_6714','c_3454']

In [ ]:
# Indication de la mention B2B ou B2C selon le tri ci-dessus
clients = ['c_1609','c_4958','c_6714','c_3454']
df_lapage['type_client']=df_lapage['client_id'].apply(lambda x: 'B to B' if x in clients else 'B to C')
df_lapage.head(20)

# Création d'un df b2b et conservation du df principal excluant ces 4 clients
df_b2b = df_lapage[df_lapage['client_id'].isin(clients)]

# Création d'un df b2b et conservation du df principal excluant ces 4 clients
df_b2c = df_lapage[df_lapage['type_client'] == 'B to C']
df_b2c

In [ ]:
# # Le fait de trier les B2B sur la quantité n'est pas cohérent car une même personne peut acheter plusieurs exemplaires d'un même ouvrage (sortie de livres, achat pour offrir ...) : 425 
df_lapage['quantity'] = (df_lapage.groupby(['id_prod', 'date', 'client_id'])['id_prod'].transform('count'))
df_lapage=df_lapage.sort_values('quantity',ascending=False)
df_clients_quantity = df_lapage.loc[df_lapage['quantity'] > 1, 'client_id'].unique()
df_clients_quantity.head()

In [ ]:
# Age des clients
df_profils_clients = df_lapage.groupby('client_id', 'categ')['birth']
df_profils_clients.head()


In [ ]:
# Top 20 des clients en terme de CA généré
df_lapage.head(20)

In [ ]:
# Flop 20 des clients en terme de CA généré
df_lapage.tail(20)